In [1]:
!pip install -U langchain langchain-openai requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.1/112.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.10
    Uninstalling langgraph-1.0.10:
      Successfully uninstalled langgraph-1.0.10
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.10
 

In [2]:
!pip install duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 35.8 MB/s eta 0:00:00


In [3]:
import os
import requests
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI

In [4]:
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
    print("OPENROUTER_API_KEY loaded successfully")
except Exception as e:
    print("Error loading API key:", e)

OPENROUTER_API_KEY loaded successfully


In [5]:
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [6]:
response = model.invoke("Answer in one short sentence: What is Ramadan?")
print(response.content)

Ramadan is the Islamic holy month of fasting, prayer, and reflection observed by Muslims worldwide.


In [12]:
import requests
from langchain.tools import tool

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10.0)
        response.raise_for_status()
        return response.text[:4000]

    except Exception as e:
        return f"Error fetching {url}: {e}"


In [13]:
from langchain.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def internet_search(query: str) -> str:
    """Search the internet and return top 3 results"""

    url = "https://duckduckgo.com/html/"
    params = {"q": query}

    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, "html.parser")

    results = []

    for a in soup.select(".result__title a", limit=3):
        title = a.get_text(strip=True)
        link = a.get("href")
        results.append(f"{title} - {link}")

    return "\n".join(results)

In [14]:
AGENT_PROMPT = """
You are an expert researcher.

Your job is to answer the user's question using information from the internet.

Steps you must follow:
1. Use internet_search to find relevant websites.
2. Select the top 3 search results.
3. Use fetch_url to retrieve the content of each website.
4. Combine the information and write a concise answer.

Always rely on multiple sources before answering.
Keep your final answer short and clear.
"""

In [15]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=AGENT_PROMPT
)

In [16]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Agentic AI?"}]}
)

print(response)


{'messages': [HumanMessage(content='What is Agentic AI?', additional_kwargs={}, response_metadata={}, id='cd6d2f38-748f-4d2b-a687-aaa7b94c8bc1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 424, 'total_tokens': 608, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 178, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 'system_fingerprint': None, 'id': 'gen-1773255287-5iVPNe43uIusEyhk69R5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cde40-6192-7fd1-9cde-7bbad2e44996-0', t